# NLP HW3 — CFEVER 中文事實查核（強化版 pipeline）

**學號：115062556　　Kaggle username：（請填寫）**

### Generative AI 使用聲明
本作業的整體架構設計與程式碼由 Anthropic **Claude（Sonnet 5.5）** 協助撰寫，內容包含：
混合式頁面檢索與學習式排序、句子 cross-encoder 重排與超連結第二跳、證據分類器與提交檔產生。
以上程式由本人在本機（RTX 3050 4GB）實際執行、檢查並依結果調整；實驗數據皆為本人實際跑出。
參考資料：作業提供的 sample code（資料讀取、評分函式、提交格式）、CFEVER 論文與資料集說明
（https://github.com/IKMLab/CFEVER-data）、Hugging Face 上的 `BAAI/bge-small-zh-v1.5` 與 `hfl/chinese-macbert-base`。

### Pipeline
1. **Document Retrieval**：標題子字串比對 + BM25（字元 bigram）+ 稠密檢索（bge-small-zh）→ 用 HistGradientBoosting 學習式排序，取前 K 頁。
2. **Evidence Retrieval**：對候選頁面的句子，用微調過的 cross-encoder（MacBERT）打分；再沿著前幾名句子的超連結做「第二跳」補充頁面。
3. **Classification**：MacBERT 以 claim + 前 4 句證據分三類（supports / refutes / NOT ENOUGH INFO）；v2 起改為**多個分類器的集成**（MacBERT-base、RoBERTa-wwm-ext，可再加 MacBERT-large），logits 取平均後再校準。

與 sample code 的主要差異：sample code 只索引「訓練集涉及的頁面」，但 dev 的證據頁面只有約 11% 出現在其中；
本版改用**完整 wiki（約 119 萬頁）**，並以廉價檢索縮小候選後才使用 BERT 類模型。

**執行方式**：先把 `QUICK_TEST = True` 跑一遍（用 `wiki_sample.jsonl`，約十分鐘）確認環境與流程無誤，再改成 `False` 做完整執行。
每個階段的結果都會存到 `output/cache/`，中途中斷後重跑會自動略過已完成的階段。

In [17]:
# ===== 1. 套件與參數 =====
# （Generative AI 使用聲明：本 notebook 程式碼由 Claude 協助撰寫，詳見上方說明。）
import os, re, json, math, time, random, pickle, gc, functools, collections
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

# ---------- 最重要的開關 ----------
QUICK_TEST = False        # True：用 wiki_sample.jsonl 與少量資料，十分鐘內跑完整條流程（先用它檢查環境）
FORCE_RETRAIN = False     # True：忽略已存的模型快取，重新訓練（包含重排器，約 40 分鐘）
FORCE_RETRAIN_CLASSIFIER = False   # True：只重新訓練分類器
SEED = 2026

# ---------- 檢索參數 ----------
LEAD_RAW_CHARS, LEAD_CHARS = 400, 110   # 頁面「摘要」= 標題 + 前 110 字（BM25 與稠密檢索使用）
BM25_FEATURES, BM25_K1, BM25_B, BM25_MAX_DF = 2 ** 21, 1.2, 0.75, 0.05
K_BM25, K_DENSE = 50, 50                # BM25 / 稠密檢索各取前幾名
CAND_BM25, CAND_DENSE = 20, 20          # 其中前幾名進入候選集
TITLE_MIN, TITLE_MAX = 2, 24            # 標題比對的長度範圍（單字標題噪音太大，直接略過）
PAGES_PER_TITLE, MAX_TITLE_CANDS = 6, 40
USE_DENSE = True
DENSE_MODEL = "BAAI/bge-small-zh-v1.5"
DENSE_MAX_LEN, DENSE_BS = 96, 256

# ---------- 句子檢索 ----------
PAGE_TOPK = 6                           # 取排序後前幾頁的句子當候選
PER_PAGE_MAX_SID = 60                   # 每頁只看前 60 句（gold 有 95% 在 50 句內）
RERANK_POOL = 32                        # 每個 claim 送進 cross-encoder 的句子數
USE_HOP2 = True                         # 是否沿超連結做第二跳
HOP_SRC, HOP_MAX_PAGES, HOP_PER_PAGE, HOP_POOL = 2, 6, 4, 16
RERANK_MODEL = "hfl/chinese-macbert-base"
RERANK_MAX_LEN, RERANK_NEG = 128, 5
RERANK_EPOCHS, RERANK_LR, RERANK_BS, RERANK_ACCUM = 1, 3e-5, 16, 2
RERANK_TRAIN_CLAIMS = None              # None = 使用全部 supports/refutes 訓練題；顯存/時間不夠可設成 8000

# ---------- 分類器 ----------
CLS_MODEL = "hfl/chinese-macbert-base"  # 若載入失敗可改成 "google-bert/bert-base-chinese"
CLS_CTX_K, CLS_MAX_LEN = 4, 256
CLS_EPOCHS, CLS_LR, CLS_BS, CLS_ACCUM = 2, 2e-5, 8, 4
# 分類器集成（v2）：多個不同骨幹/種子的分類器，logits（log-softmax）取平均。每個成員訓練完都存檔，之後重跑會直接載入。
#   - macbert_base 是 v1 已訓練好的分類器（classifier_v2），不會重訓。
#   - 想加成員就把 tag 加進 ACTIVE_MEMBERS：先跑 roberta_base（約 35 分鐘），確認有進步再加 macbert_large（約 1.5~2 小時）。
CLS_MEMBERS = [
    dict(tag="macbert_base", model="hfl/chinese-macbert-base", path="classifier_v2", max_len=CLS_MAX_LEN),
    dict(tag="roberta_base", model="hfl/chinese-roberta-wwm-ext", lr=2e-5, epochs=2, bs=8, accum=4, seed=11),
    # 先在中文 NLI 資料（CMNLI/OCNLI 等，約 100 萬筆）微調過的 checkpoint（IDEA-CCNL Erlangshen，Apache-2.0，皆 < 1B）。
    # NLI 的 (前提, 假設) 對應這裡的 (證據, claim)，所以 swap=True 把證據放前面；NLI 標籤順序是 矛盾/中立/蘊含，
    # head_perm 把分類頭重排成 supports/refutes/NEI = 蘊含/矛盾/中立，讓微調從「已經懂得推論」的起點開始。
    dict(tag="erlangshen_110m", model="IDEA-CCNL/Erlangshen-Roberta-110M-NLI", lr=2e-5, epochs=2, bs=8, accum=4, seed=21,
         swap=True, head_perm=[2, 0, 1]),
    # 330M（large）：4GB 顯存放不下完整 AdamW，改用 Adafactor + 凍結 embedding 與前 8 層（OOM 時自動多凍結 4 層再重試）。
    # 注意：不要用 Erlangshen-MegatronBert-1.3B-NLI，它超過助教規定的 1B 上限。
    dict(tag="erlangshen_330m", model="IDEA-CCNL/Erlangshen-Roberta-330M-NLI", lr=1.5e-5, epochs=2, bs=4, accum=8, seed=22,
         swap=True, head_perm=[2, 0, 1], freeze=8, optim="adafactor", infer_bs=16),
]
ACTIVE_MEMBERS = ["macbert_base", "roberta_base", "erlangshen_110m", "erlangshen_330m"]
PRIOR_TARGET = (1 / 3, 1 / 3, 1 / 3)    # 僅用於「舊的先驗修正」消融；正式預測改用 dev 校準（見分類評估 cell）
CLS_TRAIN_SR_CLAIMS = None              # None = 所有 supports/refutes 訓練題都用重排器重排一次，當作分類訓練證據（約 25 分鐘）
CLS_SHUFFLE_P = 0.3                     # 訓練時有 30% 的機率把證據順序打亂，避免模型依賴「標準證據排第一」
HOP_PENALTIES = (0.0, 1.0, 2.0, 3.0, 5.0, None)   # 第二跳候選的扣分；None = 不使用第二跳。依 dev 選擇

# ---------- 共用 ----------
GRAD_CKPT = True                        # gradient checkpointing：省顯存（約慢 30%）。顯存夠可設 False
INFER_BS = 64
N_TRAIN_LIMIT = N_DEV_LIMIT = N_TEST_LIMIT = None

if QUICK_TEST:
    N_TRAIN_LIMIT, N_DEV_LIMIT, N_TEST_LIMIT = 800, 200, 100
    RERANK_TRAIN_CLAIMS = 400
    RERANK_EPOCHS = CLS_EPOCHS = 1

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("裝置：", device, torch.cuda.get_device_name(0) if USE_AMP else "（沒有 GPU，會非常慢）")

# ---------- 路徑 ----------
IS_KAGGLE = Path("/kaggle/input").exists()
if IS_KAGGLE:
    DATA_DIR, OUTPUT_DIR = Path("/kaggle/input"), Path("/kaggle/working")
else:
    DATA_DIR = next((p for p in (Path("dataset"), Path(".")) if (p / "train.jsonl").is_file()), Path("dataset"))
    OUTPUT_DIR = Path("output")


def find_file(name):
    direct = DATA_DIR / name
    if direct.is_file():
        return direct
    hits = sorted(p for p in DATA_DIR.rglob(name) if p.is_file())
    if not hits:
        raise FileNotFoundError(f"找不到 {name}；目前搜尋目錄：{DATA_DIR.resolve()}")
    return hits[0]


TRAIN_PATH, DEV_PATH, TEST_PATH = find_file("train.jsonl"), find_file("dev.jsonl"), find_file("test.jsonl")
WIKI_PATH = find_file("wiki_sample.jsonl" if QUICK_TEST else "wiki_pages.jsonl")
CACHE_DIR = OUTPUT_DIR / ("cache_quick" if QUICK_TEST else "cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("資料夾：", DATA_DIR.resolve(), "| wiki：", WIKI_PATH.name, "| 輸出：", OUTPUT_DIR.resolve())

裝置： cuda NVIDIA GeForce RTX 3050 Laptop GPU
資料夾： /home/yiting/NLP-Assignment3/dataset | wiki： wiki_pages.jsonl | 輸出： /home/yiting/NLP-Assignment3/output


In [18]:
# ===== 2. 共用小工具 =====
_SPACE = re.compile(r"(?<=[^\x00-\x7f])\s+|\s+(?=[^\x00-\x7f])")   # 只去掉與中文字（非 ASCII）相鄰的空白
_PAREN = re.compile(r"_\(.*\)$")


def norm(s):
    """wiki 文字是斷詞過的（『數學 ， 是研究…』），claim 沒有空格；統一去掉與中文相鄰的空白。"""
    return _SPACE.sub("", s).strip()


def title_base(t):
    """'愛爾蘭_(消歧義)' -> '愛爾蘭'（用來和 claim 做子字串比對）"""
    return _PAREN.sub("", t).replace("_", " ")


def bigrams(s):
    if len(s) < 2:
        return {s} if s else set()
    return {s[i:i + 2] for i in range(len(s) - 1)}


def read_jsonl(path):
    with open(path, encoding="utf-8-sig") as f:
        return [json.loads(l) for l in f if l.strip()]


def free_gpu():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()


def amp_ctx():
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP)


def cached(path, build_fn, force=False):
    """有快取就讀，沒有就執行 build_fn 並存檔。"""
    path = Path(path)
    if path.exists() and not force:
        with open(path, "rb") as f:
            return pickle.load(f)
    obj = build_fn()
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)
    return obj


LABEL2ID = {"SUPPORTS": 0, "REFUTES": 1, "NOT ENOUGH INFO": 2}
LABEL_OUT = ["supports", "refutes", "NOT ENOUGH INFO"]   # 與 sample code 的輸出格式一致


def parse_gold(row):
    """回傳 [[(page_id, sentence_id), ...], ...]；NEI 沒有證據，回傳 []。"""
    if row["label"].upper() == "NOT ENOUGH INFO":
        return []
    groups = []
    for g in row["evidence"]:
        items = [(it[2], int(it[3])) for it in g if it[2] is not None and it[3] is not None]
        if items:
            groups.append(items)
    return groups

## 3. 讀取資料與完整 Wikipedia 索引

單次掃描 `wiki_pages.jsonl`（約 1GB）：記下每頁標題、檔案位移（之後需要句子時才回頭讀）、頁面長度，以及「標題 + 開頭」摘要。
之後所有檢索都建立在這份摘要上，所以不需要把 740 萬個句子放進記憶體。

In [19]:
def scan_wiki(path):
    titles, offsets, plen, leads = [], [], [], []
    pos = 0
    with open(path, "rb") as f:
        for raw in tqdm(f, desc="掃描 wiki", unit="頁", mininterval=5):
            n = len(raw)
            if raw.strip():
                page = json.loads(raw)
                t = page["id"]
                text = page.get("text", "") or ""
                titles.append(t)
                offsets.append(pos)
                plen.append(len(text))
                leads.append(title_base(t) + "：" + norm(text[:LEAD_RAW_CHARS])[:LEAD_CHARS])
            pos += n
    return dict(titles=titles, offsets=np.array(offsets, dtype=np.int64), plen=np.array(plen, dtype=np.int32), leads=leads)


_sig = f"{WIKI_PATH.stat().st_size}_{LEAD_CHARS}"
wiki = cached(CACHE_DIR / f"wiki_meta_{_sig}.pkl", lambda: scan_wiki(WIKI_PATH))
titles, offsets, plen, leads = wiki["titles"], wiki["offsets"], wiki["plen"], wiki["leads"]
NPAGES = len(titles)
title2idx = {t: i for i, t in enumerate(titles)}
base_titles = [title_base(t) for t in titles]
base2idx = defaultdict(list)           # '愛爾蘭' -> [頁面索引...]；標題本身（沒有括號）排最前面
for i, (t, b) in enumerate(zip(titles, base_titles)):
    if len(b) >= TITLE_MIN:
        if t.replace("_", " ") == b:
            base2idx[b].insert(0, i)
        else:
            base2idx[b].append(i)
print("頁面數：", NPAGES, "| 可比對的標題數：", len(base2idx))

WIKI_FH = open(WIKI_PATH, "rb")


@functools.lru_cache(maxsize=80000)
def get_sentences(pidx):
    """回傳 ((sentence_id, 去空白文字, 超連結目標頁面tuple), ...)；略過空句。"""
    WIKI_FH.seek(int(offsets[pidx]))
    page = json.loads(WIKI_FH.readline())
    out = []
    for line in page.get("lines", "").split("\n"):
        parts = line.split("\t")
        if len(parts) < 2 or not parts[0].isdigit():
            continue
        text = norm(parts[1])
        if text:
            out.append((int(parts[0]), text, tuple(x for x in parts[3::2] if x)))   # parts[2::2] 是錨文字，[3::2] 是目標頁面
    return tuple(out)


def disp(pidx):
    return titles[pidx].replace("_", " ")


def passage(pidx, text):
    return f"{disp(pidx)}：{text}"


print(get_sentences(0)[:2])

頁面數： 1187751 | 可比對的標題數： 1078608
((0, '數學，是研究數量、結構以及空間等概念及其變化的一門學科，屬於形式科學的一種。', ()), (2, '基礎數學的知識與運用是生活中不可或缺的一環。', ()))


In [20]:
# ----- 讀取 claim，建立 train / dev / test -----
def make_split(rows, name, labeled=True):
    s = dict(name=name, id=[r["id"] for r in rows], claim=[norm(r["claim"]) for r in rows])
    if labeled:
        s["label"] = np.array([LABEL2ID[r["label"].upper()] for r in rows])
        s["gold"] = [parse_gold(r) for r in rows]
    return s


def covered(groups):
    return any(all(t in title2idx for t, _ in g) for g in groups)


rng = random.Random(SEED)
train_rows, dev_rows, test_rows = read_jsonl(TRAIN_PATH), read_jsonl(DEV_PATH), read_jsonl(TEST_PATH)
if QUICK_TEST:   # 抽樣 wiki 只含部分頁面：只保留證據頁面都在抽樣內的題目
    ok = lambda r: r["label"].upper() == "NOT ENOUGH INFO" or covered(parse_gold(r))
    train_rows, dev_rows = [r for r in train_rows if ok(r)], [r for r in dev_rows if ok(r)]
for rows, lim in ((train_rows, N_TRAIN_LIMIT), (dev_rows, N_DEV_LIMIT), (test_rows, N_TEST_LIMIT)):
    if lim is not None and len(rows) > lim:
        rows[:] = rng.sample(rows, lim)

train = make_split(train_rows, "train")
dev = make_split(dev_rows, "dev")
test = make_split(test_rows, "test", labeled=False)
for s in (train, dev, test):
    print(s["name"], len(s["claim"]), (np.bincount(s["label"], minlength=3).tolist() if "label" in s else ""))
missing = sum(1 for s in (train, dev) for gs in s["gold"] for g in gs for t, _ in g if t not in title2idx)
print("標準答案頁面不在 wiki 的數量（完整模式應為 0；QUICK 模式因 wiki 為抽樣，可能不為 0）：", missing)

train 24012 [11085, 7113, 5814]
dev 3000 [1000, 1000, 1000]
test 6000 
標準答案頁面不在 wiki 的數量（完整模式應為 0；QUICK 模式因 wiki 為抽樣，可能不為 0）： 0


## 4. 第一階段：Document Retrieval

三種廉價訊號，各有所長：
- **標題比對**：claim 裡出現的 wiki 標題（dev 約 80% 的證據頁面標題會原封不動出現在 claim 中）。
  單字標題每題平均出現 13 個、幾乎都是噪音，所以只比對長度 ≥ 2 的標題。
- **BM25**（字元 bigram）：處理 claim 沒有直接寫出標題的情況；索引的是「標題 + 開頭 110 字」。
- **稠密檢索**（bge-small-zh）：處理改寫、同義詞。

三者取聯集當候選頁面，再用 **HistGradientBoosting** 學習怎麼排序（特徵包含標題長度、位置、是否被更長標題包含、
BM25 / 稠密檢索的分數與名次等）。標籤是「該頁是否為標準證據頁面」，只用訓練集，且訓練集的排序分數用 3 折交叉預測，避免過度樂觀。

In [21]:
# ----- BM25（字元 bigram，hashing）-----
HV = HashingVectorizer(analyzer="char", ngram_range=(2, 2), n_features=BM25_FEATURES, alternate_sign=False,
                       norm=None, lowercase=True, dtype=np.float32)


def build_bm25():
    mats = [HV.transform(leads[s:s + 100000]) for s in tqdm(range(0, NPAGES, 100000), desc="BM25 建索引")]
    X = sp.vstack(mats, format="csr")
    del mats
    df = np.bincount(X.indices, minlength=BM25_FEATURES)
    idf = np.log((NPAGES - df + 0.5) / (df + 0.5) + 1).astype(np.float32)
    idf[df > BM25_MAX_DF * NPAGES] = 0.0          # 幾乎每頁都有的 bigram（停用詞）直接忽略
    dl = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
    dl_rep = np.repeat(dl, np.diff(X.indptr))
    X.data = X.data * (BM25_K1 + 1) / (X.data + BM25_K1 * (1 - BM25_B + BM25_B * dl_rep / max(dl.mean(), 1e-6)))
    del dl_rep
    return X.tocsc(), idf


_bm25_path = CACHE_DIR / f"bm25_{_sig}.npz"
if _bm25_path.exists():
    BM25_X, BM25_IDF = sp.load_npz(_bm25_path), np.load(CACHE_DIR / f"bm25_idf_{_sig}.npy")
else:
    BM25_X, BM25_IDF = build_bm25()
    sp.save_npz(_bm25_path, BM25_X, compressed=False)
    np.save(CACHE_DIR / f"bm25_idf_{_sig}.npy", BM25_IDF)
BM25_PTR, BM25_IND, BM25_DAT = BM25_X.indptr, BM25_X.indices, BM25_X.data
print("BM25 索引：", BM25_X.shape, "非零元素：", BM25_X.nnz)


def bm25_scores(claim):
    q = HV.transform([claim])
    scores = np.zeros(NPAGES, dtype=np.float32)
    for c in q.indices:
        w = BM25_IDF[c]
        if w > 0:
            s, e = BM25_PTR[c], BM25_PTR[c + 1]
            scores[BM25_IND[s:e]] += w * BM25_DAT[s:e]
    return scores


def topk_idx(scores, k):
    k = min(k, len(scores))
    top = np.argpartition(-scores, k - 1)[:k]
    return top[np.argsort(-scores[top], kind="stable")]

BM25 索引： (1187751, 2097152) 非零元素： 84250765


In [22]:
# ----- 稠密檢索（bge-small-zh）-----
def load_dense_model():
    tok = AutoTokenizer.from_pretrained(DENSE_MODEL)
    model = AutoModel.from_pretrained(DENSE_MODEL).to(device).eval()
    if device.type == "cuda":
        model = model.half()
    return tok, model


@torch.no_grad()
def encode_dense(texts, tok, model, desc="編碼"):
    """CLS 向量 + L2 正規化，回傳 float16 numpy。依長度排序後分批，可省很多時間。"""
    order = np.argsort([len(t) for t in texts], kind="stable")
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float16)
    for s in tqdm(range(0, len(texts), DENSE_BS), desc=desc, mininterval=5):
        ids = order[s:s + DENSE_BS]
        enc = tok([texts[i] for i in ids], padding=True, truncation=True, max_length=DENSE_MAX_LEN, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        h = model(**enc).last_hidden_state[:, 0]
        out[ids] = F.normalize(h.float(), dim=-1).cpu().numpy().astype(np.float16)
    return out


@torch.no_grad()
def dense_topk(Q, P, k, chunk=200000, qbatch=512):
    """Q:[n,d]、P:[N,d]（皆 float16 numpy）。分塊上 GPU 算內積並合併 top-k，避免一次吃光 4GB 顯存。"""
    n, k = len(Q), min(k, len(P))
    dt = torch.float16 if device.type == "cuda" else torch.float32
    Qg = torch.from_numpy(np.ascontiguousarray(Q)).to(device=device, dtype=dt)
    best_v = torch.full((n, k), -2.0, device=device)
    best_i = torch.zeros((n, k), dtype=torch.long, device=device)
    for ps in tqdm(range(0, len(P), chunk), desc="稠密檢索", mininterval=5):
        Pc = torch.from_numpy(np.array(P[ps:ps + chunk])).to(device=device, dtype=dt)
        for qs in range(0, n, qbatch):
            v, i = (Qg[qs:qs + qbatch] @ Pc.T).topk(min(k, Pc.shape[0]), dim=1)
            cv = torch.cat([best_v[qs:qs + qbatch], v.float()], 1)
            ci = torch.cat([best_i[qs:qs + qbatch], i + ps], 1)
            tv, ti = cv.topk(k, dim=1)
            best_v[qs:qs + qbatch], best_i[qs:qs + qbatch] = tv, ci.gather(1, ti)
    return best_i.cpu().numpy(), best_v.cpu().numpy()


_DENSE = {}


def dense_state():
    """第一次需要稠密檢索時才載入模型；頁面向量只算一次並存檔（約 10~20 分鐘）。"""
    if not _DENSE:
        tok, mdl = load_dense_model()
        path = CACHE_DIR / f"dense_pages_{_sig}_{DENSE_MODEL.split('/')[-1]}.npy"
        if not path.exists():
            np.save(path, encode_dense(leads, tok, mdl, desc="編碼全部頁面"))
        _DENSE.update(tok=tok, mdl=mdl, P=np.load(path, mmap_mode="r"))
        print("頁面向量：", _DENSE["P"].shape)
    return _DENSE

In [23]:
# ----- 標題比對 + 候選頁面 + 特徵 -----
FEATS = ["tm", "tlen", "tfrac", "tpos", "n_tm", "tm_rank", "nested", "paren", "same_base", "digit", "ptlen",
         "bm25", "bm25_rel", "bm25_rank", "dense", "dense_rel", "dense_rank", "plen", "lead_ov"]
FI = {f: i for i, f in enumerate(FEATS)}


def title_matches(claim):
    """claim 中所有以子字串形式出現的 wiki 標題（已去括號、長度 >= 2）。"""
    L, found = len(claim), {}
    for i in range(L):
        for j in range(i + TITLE_MIN, min(L, i + TITLE_MAX) + 1):
            sub = claim[i:j]
            if sub in base2idx and sub not in found:
                found[sub] = (i, j)
    return [(b, s, e) for b, (s, e) in found.items()]


def gen_features(claims, Q, d_idx, d_val, desc):
    rows_all, pidx_all, counts = [], [], []
    for qi, claim in enumerate(tqdm(claims, desc=desc, mininterval=5)):
        sc = bm25_scores(claim)
        top = topk_idx(sc, K_BM25)
        b_rank = {int(i): r for r, i in enumerate(top)}
        bmax = float(sc[top[0]]) if len(top) else 0.0
        if USE_DENSE:
            DENSE_P = dense_state()["P"]
            dq, di, dv = Q[qi].astype(np.float32), d_idx[qi], d_val[qi]
            d_rank = {int(i): r for r, i in enumerate(di)}
            dmax = float(dv[0])
        else:
            di, d_rank, dmax = np.array([], dtype=np.int64), {}, 0.0
        tms = sorted(title_matches(claim), key=lambda x: -(x[2] - x[1]))
        n_tm = len(tms)
        tm_rank_of = {b: r for r, (b, _, _) in enumerate(tms)}
        bases = [b for b, _, _ in tms]
        nested_of = {b: int(any(o != b and b in o for o in bases)) for b in bases}
        info = {}
        for b, s, e in tms:
            for pidx in base2idx[b][:PAGES_PER_TITLE]:
                if pidx not in info and len(info) < MAX_TITLE_CANDS:
                    info[pidx] = (b, s, e)
        cset = dict.fromkeys(info)
        for i in top[:CAND_BM25]:
            cset[int(i)] = None
        for i in di[:CAND_DENSE]:
            cset[int(i)] = None
        cl = list(cset)
        cos = {}
        if USE_DENSE:
            cos = {i: float(dv[d_rank[i]]) for i in cl if i in d_rank}
            miss = [i for i in cl if i not in d_rank]
            if miss:
                vals = np.asarray(DENSE_P[np.array(miss)], dtype=np.float32) @ dq
                cos.update({i: float(v) for i, v in zip(miss, vals)})
        cb = bigrams(claim)
        den, L = max(1, len(cb)), max(1, len(claim))
        rows = []
        for i in cl:
            inf = info.get(i)
            if inf:
                b, s, e = inf
                tm, tl, tf, tp, tr, ne, dg = 1, e - s, (e - s) / L, s / L, tm_rank_of[b], nested_of[b], int(b.isdigit())
            else:
                tm, tl, tf, tp, tr, ne, dg = 0, 0, 0.0, -1.0, n_tm, 0, 0
            bm = float(sc[i])
            dn = cos.get(i, 0.0)
            rows.append([tm, tl, tf, tp, n_tm, tr, ne, int("_(" in titles[i]), len(base2idx.get(base_titles[i], ())), dg,
                         len(base_titles[i]), bm, bm / bmax if bmax > 0 else 0.0, b_rank.get(i, K_BM25),
                         dn, dn - dmax, d_rank.get(i, K_DENSE), math.log1p(plen[i]), len(cb & bigrams(leads[i])) / den])
        rows_all.append(np.asarray(rows, dtype=np.float32).reshape(-1, len(FEATS)))
        pidx_all.append(np.asarray(cl, dtype=np.int64))
        counts.append(len(cl))
    counts = np.array(counts)
    return dict(X=np.concatenate(rows_all), pidx=np.concatenate(pidx_all), counts=counts,
                starts=np.concatenate([[0], np.cumsum(counts)[:-1]]))


def retrieve_features(split):
    path = CACHE_DIR / f"feat_{_sig}_{split['name']}_{len(split['claim'])}.pkl"

    def build():
        if USE_DENSE:
            ds = dense_state()
            Q = encode_dense(split["claim"], ds["tok"], ds["mdl"], desc=f"編碼 {split['name']} claim")
            d_idx, d_val = dense_topk(Q, ds["P"], K_DENSE)
        else:
            Q = d_idx = d_val = None
        return gen_features(split["claim"], Q, d_idx, d_val, desc=f"候選頁面 {split['name']}")
    return cached(path, build)


feats = {s["name"]: retrieve_features(s) for s in (dev, test, train)}
_DENSE.pop("mdl", None)         # 稠密模型之後不再需要，釋放顯存給 BERT 訓練
free_gpu()
for k, v in feats.items():
    print(k, "候選列數：", len(v["X"]), "每題平均候選頁數：", round(len(v["X"]) / len(v["counts"]), 1))

dev 候選列數： 160680 每題平均候選頁數： 53.6
test 候選列數： 334376 每題平均候選頁數： 55.7
train 候選列數： 1279826 每題平均候選頁數： 53.3


In [24]:
# ----- 學習式排序（HistGradientBoosting）與 Recall@K -----
def page_labels(split, F_):
    y = np.zeros(len(F_["X"]), dtype=np.int8)
    for qi, groups in enumerate(split["gold"]):
        gold = {title2idx[t] for g in groups for t, _ in g if t in title2idx}
        s, c = F_["starts"][qi], F_["counts"][qi]
        y[s:s + c] = [int(p in gold) for p in F_["pidx"][s:s + c]]
    return y


def fit_ranker(X, y):
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, max_leaf_nodes=48, l2_regularization=1.0,
                                          random_state=SEED).fit(X, y)


def to_ranked(F_, scores):
    out = []
    for qi in range(len(F_["counts"])):
        s, c = F_["starts"][qi], F_["counts"][qi]
        o = np.argsort(-scores[s:s + c], kind="stable")
        out.append((F_["pidx"][s:s + c][o], scores[s:s + c][o]))
    return out


Ftr = feats["train"]
y_tr = page_labels(train, Ftr)
qi_rows = np.repeat(np.arange(len(Ftr["counts"])), Ftr["counts"])
sr_rows = np.where(train["label"][qi_rows] != 2)[0]
nei_rows = np.where(train["label"][qi_rows] == 2)[0]
ranker = fit_ranker(Ftr["X"][sr_rows], y_tr[sr_rows])
oof = np.zeros(len(Ftr["X"]), dtype=np.float32)
if len(nei_rows):
    oof[nei_rows] = ranker.predict_proba(Ftr["X"][nei_rows])[:, 1]
for tr_i, va_i in GroupKFold(n_splits=3).split(sr_rows, groups=qi_rows[sr_rows]):   # 訓練題用交叉預測，避免排序分數過度樂觀
    m = fit_ranker(Ftr["X"][sr_rows[tr_i]], y_tr[sr_rows[tr_i]])
    oof[sr_rows[va_i]] = m.predict_proba(Ftr["X"][sr_rows[va_i]])[:, 1]
ranked = {"train": to_ranked(Ftr, oof)}
for name in ("dev", "test"):
    ranked[name] = to_ranked(feats[name], ranker.predict_proba(feats[name]["X"])[:, 1])


def page_recall(split, rk, K):
    hit = n = 0
    for qi, groups in enumerate(split["gold"]):
        if not groups:
            continue
        n += 1
        top = {titles[i] for i in rk[qi][0][:K]}
        hit += any(all(t in top for t, _ in g) for g in groups)
    return hit / max(n, 1)


# 消融：各訊號單獨排序 vs. 學習式排序（候選集相同，所以 K<=20 時 BM25 / 稠密單獨排序等同於原本的 top-K）
Fd, X = feats["dev"], feats["dev"]["X"]
rrf = 1 / (60 + X[:, FI["bm25_rank"]]) + 1 / (60 + X[:, FI["dense_rank"]]) + X[:, FI["tm"]] / (60 + X[:, FI["tm_rank"]])
variants = {
    "只用標題比對": X[:, FI["tm"]] * (1 + X[:, FI["tlen"]]) + 1e-3 * X[:, FI["bm25_rel"]],
    "只用 BM25": X[:, FI["bm25"]],
    "只用稠密檢索": X[:, FI["dense"]] if USE_DENSE else np.zeros(len(X)),
    "RRF 融合（無學習）": rrf,
    "學習式排序（本方法）": None,
}
Ks = (1, 3, 5, 6, 10, 20)
page_table = {}
for name, sc in variants.items():
    rk = ranked["dev"] if sc is None else to_ranked(Fd, sc)
    page_table[name] = [page_recall(dev, rk, K) for K in Ks]
page_df = pd.DataFrame(page_table, index=[f"Recall@{K}" for K in Ks]).T
print("Dev 頁面檢索（完整證據組的所有頁面都在前 K 名才算命中，僅統計 supports/refutes）")
display(page_df.round(4)) if "display" in globals() else print(page_df.round(4))
cand_recall = page_recall(dev, to_ranked(Fd, np.ones(len(X))), 10 ** 9)
print("候選集本身的上限（Recall@全部候選）：", round(cand_recall, 4))

Dev 頁面檢索（完整證據組的所有頁面都在前 K 名才算命中，僅統計 supports/refutes）
             Recall@1  Recall@3  Recall@5  Recall@6  Recall@10  Recall@20
只用標題比對         0.5235    0.7405    0.8255    0.8385     0.8635     0.9000
只用 BM25        0.5400    0.6385    0.6725    0.6845     0.7175     0.7585
只用稠密檢索         0.5235    0.6575    0.6965    0.7135     0.7555     0.7975
RRF 融合（無學習）    0.6995    0.8290    0.8535    0.8570     0.8760     0.9220
學習式排序（本方法）     0.7915    0.8955    0.9175    0.9245     0.9375     0.9410
候選集本身的上限（Recall@全部候選）： 0.9425


## 5. 第二階段：Evidence Retrieval

對每個 claim，取排序後前 `PAGE_TOPK` 頁的句子（先以字元 bigram 重疊度粗篩到 32 句），再交給**微調過的 cross-encoder**（MacBERT）打分。
輸入是 `claim [SEP] 頁面標題：句子`，把標題放進去是因為很多句子以「他」「該」開頭，沒有主詞。

**第二跳**：CFEVER 約 10% 的題目需要兩個以上的頁面，且第二個頁面常常沒有出現在 claim 中，而是被第一個證據句「連結」過去
（`lines` 欄位在句子文字之後附有（錨文字, 目標頁面）成對資料）。因此取第一輪分數最高的前 2 句，把它們連結的頁面（最多 6 頁）的候選句也一起打分。

訓練資料：正例是標準證據句；負例是同頁面其他句子、被檢索到的頁面的其他句子、以及標準證據句所連結頁面的句子（最難的負例，教模型區分第二跳）。

In [25]:
def build_pool(claim, page_idx, page_prob, pool_size):
    """回傳依粗篩分數排序的候選句 [(pre, pidx, sid, text, links), ...]。"""
    cb = bigrams(claim)
    den = max(1, len(cb))
    cands = []
    for pidx, pp in zip(page_idx, page_prob):
        for sid, text, links in get_sentences(int(pidx)):
            if sid >= PER_PAGE_MAX_SID:
                continue
            cands.append((len(cb & bigrams(text)) / den + 0.5 * float(pp) + (0.1 if sid == 0 else 0.0), int(pidx), sid, text, links))
    cands.sort(key=lambda c: -c[0])
    return cands[:pool_size]


def extra_pool(claim, pages, per_page, exclude=()):
    """指定頁面中，粗篩分數最高的句子（用於第二跳與訓練負例）。"""
    cb = bigrams(claim)
    den = max(1, len(cb))
    cands = []
    for pidx in pages:
        sents = [(len(cb & bigrams(t)) / den + (0.1 if sid == 0 else 0.0), int(pidx), sid, t, lk)
                 for sid, t, lk in get_sentences(int(pidx)) if sid < PER_PAGE_MAX_SID and (titles[int(pidx)], sid) not in exclude]
        sents.sort(key=lambda c: -c[0])
        cands.extend(sents[:per_page])
    cands.sort(key=lambda c: -c[0])
    return cands


# ----- 通用：微調序列分類模型 / 推論 -----
def _new_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def train_seq_clf(examples, num_labels, model_name, max_len, epochs, lr, micro_bs, accum, desc, seed=None, freeze=0, optim="adamw", head_perm=None):
    """examples: [(text_a, text_b, label), ...]。標準流程：AdamW + 線性 warmup/decay + fp16 + 梯度累積。"""
    seed = SEED if seed is None else seed
    torch.manual_seed(seed)                                   # 不同種子 → 分類頭初始化與 dropout 不同（集成需要多樣性）
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)
    if head_perm is not None and hasattr(model, "classifier") and model.classifier.out_features == len(head_perm):
        with torch.no_grad():                                 # 把 NLI 分類頭的類別順序換成我們的順序
            pm = torch.tensor(head_perm, device=model.classifier.weight.device)
            model.classifier.weight.copy_(model.classifier.weight[pm].clone())
            model.classifier.bias.copy_(model.classifier.bias[pm].clone())
    if freeze:                                                # 凍結 embedding 與前 freeze 層：省顯存、也省反向傳播時間
        base = model.base_model
        for p in base.embeddings.parameters():
            p.requires_grad = False
        for layer in base.encoder.layer[:freeze]:
            for p in layer.parameters():
                p.requires_grad = False
    if GRAD_CKPT:
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            model.gradient_checkpointing_enable()
    params = [p for p in model.parameters() if p.requires_grad]
    if optim == "adafactor":                                  # 優化器狀態極小（無動量、二階矩分解），large 模型才放得進 4GB
        from transformers.optimization import Adafactor
        opt = Adafactor(params, lr=lr, scale_parameter=False, relative_step=False, warmup_init=False, beta1=None, weight_decay=0.0)
    else:
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    steps = math.ceil(len(examples) / (micro_bs * accum)) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = _new_scaler()
    rg = random.Random(seed)
    for ep in range(epochs):
        model.train()
        order = list(range(len(examples)))
        rg.shuffle(order)
        pbar = tqdm(range(0, len(order), micro_bs), desc=f"{desc} epoch {ep + 1}/{epochs}", mininterval=10)
        run_loss, pending = 0.0, 0
        for bi, s in enumerate(pbar):
            batch = [examples[i] for i in order[s:s + micro_bs]]
            enc = tok([b[0] for b in batch], [b[1] for b in batch], truncation=True, max_length=max_len,
                      padding=True, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            labels = torch.tensor([b[2] for b in batch], device=device)
            with amp_ctx():
                logits = model(**enc).logits
            loss = F.cross_entropy(logits.float(), labels) / accum
            scaler.scale(loss).backward()
            run_loss = 0.98 * run_loss + 0.02 * loss.item() * accum if bi else loss.item() * accum
            pending += 1
            if pending == accum or s + micro_bs >= len(order):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
                pending = 0
            if bi % 20 == 0:
                pbar.set_postfix(loss=round(run_loss, 4))
    model.eval()
    return model, tok


@torch.no_grad()
def score_pairs(bundle, pairs, max_len, bs, desc="推論"):
    """pairs: [(text_a, text_b), ...] -> logits [n, num_labels]（依長度排序分批以加速）。"""
    model, tok = bundle
    model.eval()
    order = np.argsort([len(a) + len(b) for a, b in pairs], kind="stable")
    out = None
    for s in tqdm(range(0, len(order), bs), desc=desc, mininterval=10):
        ids = order[s:s + bs]
        enc = tok([pairs[i][0] for i in ids], [pairs[i][1] for i in ids], truncation=True, max_length=max_len,
                  padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with amp_ctx():
            lg = model(**enc).logits.float().cpu().numpy()
        if out is None:
            out = np.zeros((len(pairs), lg.shape[1]), dtype=np.float32)
        out[ids] = lg
    return out if out is not None else np.zeros((0, 2), dtype=np.float32)


def offload(bundle):
    """把不再使用的模型移到 CPU，釋放 4GB 顯存給下一階段的訓練。"""
    bundle[0].to("cpu")
    free_gpu()


def save_bundle(bundle, path):
    bundle[0].save_pretrained(path)
    bundle[1].save_pretrained(path)


def load_bundle(path):
    return (AutoModelForSequenceClassification.from_pretrained(path).to(device).eval(), AutoTokenizer.from_pretrained(path))

In [26]:
# ----- 訓練 cross-encoder 重排器 -----
def build_rerank_examples(split, rk, claim_ids):
    rg = random.Random(SEED)
    ex = []
    for qi in tqdm(claim_ids, desc="建立重排訓練資料", mininterval=5):
        groups = split["gold"][qi]
        claim = split["claim"][qi]
        gold = {(t, s) for g in groups for t, s in g}
        pos, gold_pages = [], []
        for t, s in gold:
            j = title2idx.get(t)
            if j is None:
                continue
            gold_pages.append(j)
            for sid, text, links in get_sentences(j):
                if sid == s:
                    pos.append((claim, passage(j, text), 1))
        if not pos:
            continue
        idxs, probs = rk[qi]
        pool = build_pool(claim, idxs[:PAGE_TOPK], probs[:PAGE_TOPK], 40)
        negs = {(c[1], c[2]): c for c in pool if (titles[c[1]], c[2]) not in gold}
        for c in extra_pool(claim, set(gold_pages), 6, exclude=gold):                     # 同一個標準頁面的其他句子
            negs.setdefault((c[1], c[2]), c)
        hop_pages = set()
        for t, s in gold:
            j = title2idx.get(t)
            for sid, _, links in (get_sentences(j) if j is not None else ()):
                if sid == s:
                    hop_pages.update(title2idx[l] for l in links if l in title2idx)
        for c in extra_pool(claim, hop_pages - set(gold_pages), 3, exclude=gold)[:6]:      # 標準證據句連結到的頁面（第二跳負例）
            negs.setdefault((c[1], c[2]), c)
        neg_list = sorted(negs.values(), key=lambda c: -c[0])
        chosen = neg_list[:3] + rg.sample(neg_list[3:], min(RERANK_NEG - 3, len(neg_list[3:])))
        ex.extend(pos)
        ex.extend((claim, passage(c[1], c[3]), 0) for c in chosen)
    rg.shuffle(ex)
    return ex


_rr_path = CACHE_DIR / "reranker"
if _rr_path.exists() and not FORCE_RETRAIN:
    reranker = load_bundle(_rr_path)
    print("載入已訓練的重排器：", _rr_path)
else:
    sr_ids = [i for i in range(len(train["claim"])) if train["label"][i] != 2]
    random.Random(SEED).shuffle(sr_ids)
    sr_ids = sr_ids[:RERANK_TRAIN_CLAIMS] if RERANK_TRAIN_CLAIMS else sr_ids
    rr_examples = build_rerank_examples(train, ranked["train"], sr_ids)
    print("重排訓練資料：", len(rr_examples), "筆（正例", sum(e[2] for e in rr_examples), "）")
    reranker = train_seq_clf(rr_examples, 2, RERANK_MODEL, RERANK_MAX_LEN, RERANK_EPOCHS, RERANK_LR, RERANK_BS, RERANK_ACCUM, "訓練重排器")
    save_bundle(reranker, _rr_path)
    del rr_examples
free_gpu()
RR_TAG = str(int((_rr_path / "config.json").stat().st_mtime))   # 重排器重訓後，舊的重排快取自動失效

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 472.34it/s]


載入已訓練的重排器： output/cache/reranker


In [27]:
# ----- 重排推論（含第二跳）-----
def rerank_split(split, rk, desc):
    """回傳 (first, final)：first 只含第一跳；final 加上第二跳。每題是 [(score, pidx, sid, text, links), ...] 由高到低。"""
    n = len(split["claim"])
    pools = [build_pool(split["claim"][qi], rk[qi][0][:PAGE_TOPK], rk[qi][1][:PAGE_TOPK], RERANK_POOL) for qi in range(n)]
    pairs = [(split["claim"][qi], passage(c[1], c[3])) for qi in range(n) for c in pools[qi]]
    own = [(qi, k) for qi in range(n) for k in range(len(pools[qi]))]
    lg = score_pairs(reranker, pairs, RERANK_MAX_LEN, INFER_BS, desc)
    first = [[] for _ in range(n)]
    sc = (lg[:, 1] - lg[:, 0]) if len(lg) else np.zeros(0)
    for (qi, k), v in zip(own, sc):
        c = pools[qi][k]
        first[qi].append((float(v), c[1], c[2], c[3], c[4]))
    first = [sorted(x, key=lambda t: -t[0]) for x in first]
    if not USE_HOP2:
        return first, first
    hop_pools, hop_pairs, hop_own = [], [], []
    for qi in range(n):
        have = {c[1] for c in pools[qi]}
        targets = []
        for _, pidx, sid, text, links in first[qi][:HOP_SRC]:
            for t in links:
                j = title2idx.get(t)
                if j is not None and j not in have and j not in targets:
                    targets.append(j)
        hp = extra_pool(split["claim"][qi], targets[:HOP_MAX_PAGES], HOP_PER_PAGE)[:HOP_POOL]
        hop_pools.append(hp)
        for k, c in enumerate(hp):
            hop_pairs.append((split["claim"][qi], passage(c[1], c[3])))
            hop_own.append((qi, k))
    final = [list(x) for x in first]
    if hop_pairs:
        hl = score_pairs(reranker, hop_pairs, RERANK_MAX_LEN, INFER_BS, desc + "（第二跳）")
        for (qi, k), v in zip(hop_own, hl[:, 1] - hl[:, 0]):
            c = hop_pools[qi][k]
            final[qi].append((float(v), c[1], c[2], c[3], c[4]))
    final = [sorted(x, key=lambda t: -t[0]) for x in final]
    return first, final


def evidence_recall(split, lists, K=5):
    hit = n = 0
    for qi, groups in enumerate(split["gold"]):
        if not groups:
            continue
        n += 1
        top = {(titles[c[1]], c[2]) for c in lists[qi][:K]}
        hit += any(all(x in top for x in g) for g in groups)
    return hit / max(n, 1)


rr = {}
for s in (dev, test):
    rr[s["name"]] = cached(CACHE_DIR / f"rerank_{_sig}_{RR_TAG}_{s['name']}_{len(s['claim'])}.pkl",
                           lambda s=s: rerank_split(s, ranked[s["name"]], f"重排 {s['name']}"))


def merge_hop(first, final, delta):
    """第二跳候選（final 中不在 first 的句子）扣掉 delta 分後與第一跳合併；delta=None 代表不使用第二跳。"""
    if delta is None:
        return first
    seen = {(c[1], c[2]) for c in first}
    out = list(first) + [(c[0] - delta,) + tuple(c[1:]) for c in final if (c[1], c[2]) not in seen]
    return sorted(out, key=lambda t: -t[0])


ev_rows = {}
for d in HOP_PENALTIES:
    lists = [merge_hop(a, b, d) for a, b in zip(*rr["dev"])]
    ev_rows["不使用第二跳" if d is None else f"第二跳（扣 {d:g} 分）"] = [evidence_recall(dev, lists, K) for K in (1, 3, 5)]
ev_df = pd.DataFrame(ev_rows, index=["Recall@1", "Recall@3", "Recall@5"]).T
print("Dev 證據句檢索（前 K 句需完整涵蓋任一組標準證據；僅統計 supports/refutes）")
display(ev_df.round(4)) if "display" in globals() else print(ev_df.round(4))

# 第二跳的句子容易擠掉第一跳的正確句子，所以只有在 Recall@5 明顯提升（>= 0.2 個百分點）時才採用
HOP_DELTA, _best = None, evidence_recall(dev, rr["dev"][0], 5)
for d in HOP_PENALTIES:
    if d is not None and ev_rows[f"第二跳（扣 {d:g} 分）"][2] > _best + 0.002:
        HOP_DELTA, _best = d, ev_rows[f"第二跳（扣 {d:g} 分）"][2]
print("採用的第二跳扣分：", HOP_DELTA, "（None = 不使用第二跳）")
EV = {n: [merge_hop(a, b, HOP_DELTA) for a, b in zip(*rr[n])] for n in ("dev", "test")}   # 之後都使用這份證據排序
for ng in (1, 2, 3):   # 依標準證據組的句子數分析
    ids = [i for i, gs in enumerate(dev["gold"]) if gs and min(len(g) for g in gs) == ng]
    if ids:
        sub = dict(gold=[dev["gold"][i] for i in ids])
        print(f"  最短證據組含 {ng} 句：{len(ids)} 題，Recall@5 =", round(evidence_recall(sub, [EV["dev"][i] for i in ids], 5), 4))

Dev 證據句檢索（前 K 句需完整涵蓋任一組標準證據；僅統計 supports/refutes）
            Recall@1  Recall@3  Recall@5
第二跳（扣 0 分）    0.4885    0.7865    0.8540
第二跳（扣 1 分）    0.4945    0.7965    0.8630
第二跳（扣 2 分）    0.4955    0.8015    0.8635
第二跳（扣 3 分）    0.4955    0.8020    0.8635
第二跳（扣 5 分）    0.4955    0.8015    0.8635
不使用第二跳        0.4955    0.8005    0.8615
採用的第二跳扣分： None （None = 不使用第二跳）
  最短證據組含 1 句：1273 題，Recall@5 = 0.9356
  最短證據組含 2 句：513 題，Recall@5 = 0.7856
  最短證據組含 3 句：169 題，Recall@5 = 0.6509


## 6. 第三階段：Classification

輸入 `[CLS] claim [SEP] 標題：證據1 [SEP] 標題：證據2 … [SEP]`（前 4 句，依重排分數由高到低；97% 的標準證據組不超過 3 句），MacBERT 分三類。

訓練資料的證據要貼近測試時的分佈：
**關鍵：每一類的訓練輸入都必須長得跟測試時一樣**，否則模型會學到與標籤無關的捷徑。第一版（v0）犯了這個錯：
NEI 的證據永遠是 4 句，但 supports/refutes 是「標準證據 + 隨機 0~3 句干擾」，大約一半不足 4 句，而且標準證據永遠排第一。
結果模型學成「滿 4 句 = NEI」，測試時每題都是 4 句，於是 dev 上 43% 的 supports/refutes 被判成 NEI。

現在三類全部用**同一個重排器對訓練題實際重排的結果**建立證據：
- **所有題目**：取重排後的前 4 句，依重排分數排序（有 30% 機率打亂順序）。
- **supports / refutes**：若標準證據不在前 4 句內，就把缺的標準證據句放進去、擠掉分數最低的非標準句，確保這題「看得到證據」。
- **NEI**：沒有標準證據，直接用重排器挑出的前 4 句。

最後再用 dev 的 logits 校準各類別的偏差（只有 2 個參數，並用二折交叉驗證確認真的有效才採用）。

In [28]:
def ctx_text(items):
    return " [SEP] ".join(passage(p, t) for p, _, t in items)


def build_cls_examples(seed=None):
    rg = random.Random(SEED if seed is None else seed)
    nei_ids = [i for i in range(len(train["claim"])) if train["label"][i] == 2]
    sr_ids = [i for i in range(len(train["claim"])) if train["label"][i] != 2]
    if CLS_TRAIN_SR_CLAIMS:
        rg.shuffle(sr_ids)
        sr_ids = sorted(sr_ids[:CLS_TRAIN_SR_CLAIMS])

    def rerank_ids(ids, tag):
        sub = dict(claim=[train["claim"][i] for i in ids])
        first, final = cached(CACHE_DIR / f"rerank_{_sig}_{RR_TAG}_train{tag}_{len(ids)}.pkl",
                              lambda: rerank_split(sub, [ranked["train"][i] for i in ids], f"重排 train-{tag}"))
        return [merge_hop(a, b, HOP_DELTA) for a, b in zip(first, final)]

    nei_lists, sr_lists = rerank_ids(nei_ids, "NEI"), rerank_ids(sr_ids, "SR")
    ex = []
    for pos_, qi in enumerate(tqdm(sr_ids, desc="建立分類訓練資料(S/R)", mininterval=5)):
        lst = sr_lists[pos_]
        groups = train["gold"][qi]
        best = min(groups, key=lambda g: (len({t for t, _ in g}), len(g)))
        by_key = {(titles[c[1]], c[2]): c for c in lst}
        top_score = lst[0][0] if lst else 0.0
        gold_items = []
        for t, s in best:
            if (t, s) in by_key:
                gold_items.append(by_key[(t, s)])
                continue
            j = title2idx.get(t)                       # 標準證據不在候選池：以最高分注入，確保這題看得到證據
            txt = next((x for sid, x, _ in (get_sentences(j) if j is not None else ()) if sid == s), None)
            if txt is None:
                gold_items = None
                break
            gold_items.append((top_score, j, s, txt, ()))
        if not gold_items:
            continue
        all_gold = {(t, s) for g in groups for t, s in g}
        fill = [c for c in lst if (titles[c[1]], c[2]) not in all_gold][:max(0, CLS_CTX_K - len(gold_items))]
        ctx = sorted(gold_items + fill, key=lambda c: -c[0])[:CLS_CTX_K]
        if rg.random() < CLS_SHUFFLE_P:
            rg.shuffle(ctx)
        ex.append((train["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in ctx]), int(train["label"][qi])))
    for pos_, qi in enumerate(nei_ids):
        ctx = list(nei_lists[pos_][:CLS_CTX_K])
        if ctx:
            if rg.random() < CLS_SHUFFLE_P:
                rg.shuffle(ctx)
            ex.append((train["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in ctx]), 2))
    rg.shuffle(ex)
    n_ctx = collections.Counter(e[1].count(" [SEP] ") + 1 for e in ex)
    print("每筆訓練資料的證據句數分佈（三類應該一致）：", dict(sorted(n_ctx.items())))
    return ex


_cls_path = CACHE_DIR / "classifier_v2"          # v2：修正訓練/測試證據分佈不一致的問題（v0 的 classifier 不再使用）
if _cls_path.exists() and not (FORCE_RETRAIN or FORCE_RETRAIN_CLASSIFIER):
    classifier = load_bundle(_cls_path)
    cls_prior = np.array(json.loads((_cls_path / "prior.json").read_text()))
    print("載入已訓練的分類器：", _cls_path)
else:
    cls_examples = build_cls_examples()
    offload(reranker)                      # 重排器之後不再需要，釋放顯存
    cls_prior = np.bincount([e[2] for e in cls_examples], minlength=3) / len(cls_examples)
    print("分類訓練資料：", len(cls_examples), "筆，類別比例 supports/refutes/NEI =", cls_prior.round(3))
    classifier = train_seq_clf(cls_examples, 3, CLS_MODEL, CLS_MAX_LEN, CLS_EPOCHS, CLS_LR, CLS_BS, CLS_ACCUM, "訓練分類器")
    save_bundle(classifier, _cls_path)
    (_cls_path / "prior.json").write_text(json.dumps(cls_prior.tolist()))
    del cls_examples
offload(classifier)                        # 之後每個成員都是「用到才從硬碟載入」，一次只佔一個模型的顯存
free_gpu()

Loading weights:  26%|██▌       | 52/201 [00:00<00:00, 266.55it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 432.37it/s]


載入已訓練的分類器： output/cache/classifier_v2


In [29]:
# ----- 分類器集成：成員訓練、逐成員推論（logits 有快取）-----
offload(reranker)                          # 重排器之後不再使用（訓練資料的重排結果都已快取）
free_gpu()
# 先看 claim + 前 4 句證據的 token 數：若常超過 256，會把後面的證據截掉（新成員會自動改用 384）
try:
    _tk = AutoTokenizer.from_pretrained(CLS_MODEL)
    _tk.model_max_length = 100000
    _dp = [(dev["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in EV["dev"][qi][:CLS_CTX_K]])) for qi in range(len(dev["claim"]))]
    _ln = np.array([len(x) for x in _tk([a for a, _ in _dp], [b for _, b in _dp], truncation=False)["input_ids"]])
    TRUNC_RATE = float((_ln > CLS_MAX_LEN).mean())
    print(f"dev 的輸入長度（token）：中位數 {np.median(_ln):.0f}、90 百分位 {np.percentile(_ln, 90):.0f}；超過 {CLS_MAX_LEN} 而被截斷的比例 = {TRUNC_RATE:.3f}")
except Exception as e:
    print("（略過輸入長度檢查：", repr(e)[:80], "）")
    TRUNC_RATE = 0.0
EXT_LEN = 384 if TRUNC_RATE > 0.10 else CLS_MAX_LEN        # 新成員使用的最大長度
OOM_ERR = getattr(torch.cuda, "OutOfMemoryError", RuntimeError)


def member_path(m):
    return CACHE_DIR / (m.get("path") or f"cls_{m['tag']}")


def train_member(m):
    path = member_path(m)
    if path.exists() and not (FORCE_RETRAIN or FORCE_RETRAIN_CLASSIFIER):
        return
    ex = build_cls_examples(seed=m.get("seed"))
    if m.get("swap"):
        ex = [(b, a, y) for a, b, y in ex]                    # (證據, claim)：NLI 的 前提/假設 順序
    freeze, max_len = m.get("freeze", 0), m.get("max_len", EXT_LEN)
    while True:
        oom, bundle = False, None
        try:
            bundle = train_seq_clf(ex, 3, m["model"], max_len, m.get("epochs", CLS_EPOCHS), m.get("lr", CLS_LR), m.get("bs", CLS_BS),
                                   m.get("accum", CLS_ACCUM), f"訓練 {m['tag']}", seed=m.get("seed"), freeze=freeze, optim=m.get("optim", "adamw"), head_perm=m.get("head_perm"))
        except OOM_ERR:
            oom = True
        if not oom:
            break
        gc.collect(); free_gpu()
        freeze += 4
        if freeze > 16:
            raise RuntimeError(f"{m['tag']}：顯存不足，已凍結到第 16 層仍然放不下；請改用較小的模型或把 bs 降為 2")
        print(f"  顯存不足 → 改為凍結前 {freeze} 層後重試")
    save_bundle(bundle, path)
    (path / "meta.json").write_text(json.dumps(dict(max_len=max_len, freeze=freeze, swap=bool(m.get("swap")))))
    offload(bundle); del bundle, ex
    free_gpu()


for _m in CLS_MEMBERS:
    if _m["tag"] in ACTIVE_MEMBERS:
        train_member(_m)


def member_logits(m, split, lists):
    path = member_path(m)
    meta = json.loads((path / "meta.json").read_text()) if (path / "meta.json").exists() else {}
    max_len = meta.get("max_len", m.get("max_len", CLS_MAX_LEN))
    cache = None
    if split["name"] in ("dev", "test"):
        cache = CACHE_DIR / f"cls_logits_{m['tag']}_{int((path / 'config.json').stat().st_mtime)}_{RR_TAG}_{HOP_DELTA}_{split['name']}.npy"
        if cache.exists():
            return np.load(cache)
    pairs = [(split["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in lists[qi][:CLS_CTX_K]])) for qi in range(len(split["claim"]))]
    if meta.get("swap"):
        pairs = [(b, a) for a, b in pairs]
    bundle = load_bundle(path)
    lg = score_pairs(bundle, pairs, max_len, m.get("infer_bs", INFER_BS), f"分類 {split['name']} / {m['tag']}")
    offload(bundle); del bundle
    free_gpu()
    if cache is not None:
        np.save(cache, lg)
    return lg


def log_softmax(x):
    x = x - x.max(1, keepdims=True)
    return x - np.log(np.exp(x).sum(1, keepdims=True))

dev 的輸入長度（token）：中位數 253、90 百分位 357；超過 256 而被截斷的比例 = 0.485


建立分類訓練資料(S/R): 100%|██████████| 18198/18198 [00:00<00:00, 30536.20it/s]


每筆訓練資料的證據句數分佈（三類應該一致）： {4: 24012}


Writing model shards: 100%|██████████| 1/1 [00:13<00:00, 13.79s/it]


In [30]:
# ----- Dev 評估：分類準確率、CFEVER Score -----
MEMBERS = [m for m in CLS_MEMBERS if m["tag"] in ACTIVE_MEMBERS]


def classify(split, final_lists, members=None):
    """集成：各成員 log-softmax 的平均（之後的偏差校準直接加在這個分數上）。"""
    ms = members or MEMBERS
    return np.mean([log_softmax(member_logits(m, split, final_lists)) for m in ms], axis=0)


def predict_with_prior(logits, correct):
    if correct:
        logits = logits + np.log(np.array(PRIOR_TARGET) / np.maximum(cls_prior, 1e-6))
    return logits.argmax(1)


def cfever_score(split, pred, lists):
    hit = 0
    for qi in range(len(split["claim"])):
        if pred[qi] != split["label"][qi]:
            continue
        if split["label"][qi] == 2:
            hit += 1
            continue
        top = {(titles[c[1]], c[2]) for c in lists[qi][:5]}
        hit += any(all(x in top for x in g) for g in split["gold"][qi])
    return hit / len(split["claim"])


dev_logits = classify(dev, EV["dev"])
dev_y = dev["label"]
dev_ev_ok = np.array([lab == 2 or any(all(x in {(titles[c[1]], c[2]) for c in EV["dev"][qi][:5]} for x in g) for g in dev["gold"][qi])
                      for qi, lab in enumerate(dev_y)])      # NEI 不看證據；S/R 需要前 5 句涵蓋任一組標準證據


def cf(pred, y, ok):
    return float(((pred == y) & ok).mean())


member_acc = {m["tag"]: float((member_logits(m, dev, EV["dev"]).argmax(1) == dev_y).mean()) for m in MEMBERS}
print("各成員 dev accuracy（未校準）：", {k: round(v, 4) for k, v in member_acc.items()}, "| 集成：", round(float((dev_logits.argmax(1) == dev_y).mean()), 4))
acc_plain = float((dev_logits.argmax(1) == dev_y).mean())
acc_prior = float((predict_with_prior(dev_logits, True) == dev_y).mean())   # 舊做法（類別先驗修正），僅作消融
cf_plain = cf(dev_logits.argmax(1), dev_y, dev_ev_ok)
# 集成成員的所有組合（未校準）：看哪些成員真的有貢獻
import itertools
_ml = {m["tag"]: log_softmax(member_logits(m, dev, EV["dev"])) for m in MEMBERS}
subset_rows = []
for r_ in range(1, len(MEMBERS) + 1):
    for comb in itertools.combinations(_ml, r_):
        lg = np.mean([_ml[t] for t in comb], axis=0)
        subset_rows.append(dict(members="+".join(comb), acc=float((lg.argmax(1) == dev_y).mean()), cfever=cf(lg.argmax(1), dev_y, dev_ev_ok)))
subset_df = pd.DataFrame(subset_rows).sort_values("cfever", ascending=False).reset_index(drop=True)
print("成員組合的 dev 表現（未校準）：")
print(subset_df.round(4).to_string(index=False))

# ----- 校準：替 refutes、NEI 加上偏差，直接最大化 dev 的 CFEVER Score（只有 2 個參數）-----
GRID = np.arange(-4, 4.01, 0.25)


def fit_bias(logits, y, ok):
    best = (-1.0, 0.0, 0.0)
    for b1 in GRID:
        for b2 in GRID:
            s = cf((logits + np.array([0.0, b1, b2])).argmax(1), y, ok) - 1e-4 * (abs(b1) + abs(b2))   # 同分時偏好較小的偏差
            if s > best[0]:
                best = (s, b1, b2)
    return np.array([0.0, best[1], best[2]])


perm = np.random.default_rng(SEED).permutation(len(dev_y))
halves = [perm[:len(perm) // 2], perm[len(perm) // 2:]]
cross = []     # 二折交叉驗證：用一半 dev 找偏差、在另一半驗證，估計「校準真正帶來的提升」
for a, b in ((0, 1), (1, 0)):
    bias = fit_bias(dev_logits[halves[a]], dev_y[halves[a]], dev_ev_ok[halves[a]])
    cross.append((cf((dev_logits[halves[b]] + bias).argmax(1), dev_y[halves[b]], dev_ev_ok[halves[b]]),
                  cf(dev_logits[halves[b]].argmax(1), dev_y[halves[b]], dev_ev_ok[halves[b]])))
cf_cross, cf_cross_base = float(np.mean([c[0] for c in cross])), float(np.mean([c[1] for c in cross]))
USE_CAL = cf_cross > cf_cross_base + 0.005          # 交叉驗證至少提升 0.5 個百分點才採用
CAL_BIAS = fit_bias(dev_logits, dev_y, dev_ev_ok) if USE_CAL else np.zeros(3)
print(f"校準：交叉驗證 CFEVER {cf_cross_base:.4f} -> {cf_cross:.4f}；採用校準 = {USE_CAL}；偏差(supports/refutes/NEI) = {CAL_BIAS}")
dev_pred = (dev_logits + CAL_BIAS).argmax(1)
assert abs(cfever_score(dev, dev_pred, EV["dev"]) - cf(dev_pred, dev_y, dev_ev_ok)) < 1e-9   # 與前面的 CFEVER 公式一致

# 上限分析：若檢索完美（S/R 直接給標準證據），分類器能到多少？
gold_lists = []
for qi in range(len(dev["claim"])):
    if dev["label"][qi] == 2:
        gold_lists.append(EV["dev"][qi])
    else:
        g = min(dev["gold"][qi], key=lambda g: (len({t for t, _ in g}), len(g)))
        items = []
        for t, s in g:
            j = title2idx.get(t)
            items += [(0.0, j, sid, text, ()) for sid, text, _ in (get_sentences(j) if j is not None else ()) if sid == s]
        all_gold = {(t, s) for gg in dev["gold"][qi] for t, s in gg}
        fill = [c for c in EV["dev"][qi] if (titles[c[1]], c[2]) not in all_gold][:max(0, CLS_CTX_K - len(items))]   # 補滿 4 句，與訓練時的輸入長度一致
        gold_lists.append(items + fill)
oracle_logits = classify(dict(claim=dev["claim"], name="dev-oracle"), gold_lists)
acc_oracle = float((oracle_logits.argmax(1) == dev_y).mean())

metrics = dict(
    page_recall_at5=page_recall(dev, ranked["dev"], 5), page_recall_at_topk=page_recall(dev, ranked["dev"], PAGE_TOPK),
    evidence_recall_at5=evidence_recall(dev, EV["dev"], 5), evidence_recall_at5_no_hop=evidence_recall(dev, rr["dev"][0], 5), hop_delta=HOP_DELTA,
    classification_acc_no_calibration=acc_plain, classification_acc_old_prior_correction=acc_prior,
    classification_acc_calibrated=float((dev_pred == dev_y).mean()), classification_acc_oracle_evidence=acc_oracle,
    active_members=ACTIVE_MEMBERS, member_dev_acc=member_acc, ensemble_subsets=subset_df.round(4).to_dict("records"),
    cfever_score_no_calibration=cf_plain, cfever_score_calibration_crossfit=cf_cross, cfever_score_crossfit_baseline=cf_cross_base,
    use_calibration=bool(USE_CAL), calibration_bias=CAL_BIAS.tolist(), cfever_score=cf(dev_pred, dev_y, dev_ev_ok),
)
per_label = {LABEL_OUT[k]: float((dev_pred[dev_y == k] == k).mean()) for k in range(3)}
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("各類別 recall（最終預測）：", {k: round(v, 4) for k, v in per_label.items()})
print("混淆矩陣（列=真實，欄=預測；supports/refutes/NEI），未校準：")
print(pd.crosstab(pd.Series(dev_y, name="true"), pd.Series(dev_logits.argmax(1), name="pred")))
if USE_CAL:
    print("校準後：")
    print(pd.crosstab(pd.Series(dev_y, name="true"), pd.Series(dev_pred, name="pred")))

分類 dev / erlangshen_330m:   0%|          | 0/188 [00:00<?, ?it/s]

分類 dev / erlangshen_330m: 100%|██████████| 188/188 [01:13<00:00,  2.57it/s]


各成員 dev accuracy（未校準）： {'macbert_base': 0.7137, 'roberta_base': 0.7227, 'erlangshen_110m': 0.747, 'erlangshen_330m': 0.7663} | 集成： 0.7723
成員組合的 dev 表現（未校準）：
                                                  members    acc  cfever
             roberta_base+erlangshen_110m+erlangshen_330m 0.7783  0.7327
             macbert_base+erlangshen_110m+erlangshen_330m 0.7760  0.7300
                          erlangshen_110m+erlangshen_330m 0.7763  0.7280
macbert_base+roberta_base+erlangshen_110m+erlangshen_330m 0.7723  0.7280
                             roberta_base+erlangshen_330m 0.7727  0.7267
                macbert_base+roberta_base+erlangshen_330m 0.7683  0.7247
                             macbert_base+erlangshen_330m 0.7710  0.7247
                                          erlangshen_330m 0.7663  0.7183
                macbert_base+roberta_base+erlangshen_110m 0.7573  0.7130
                             roberta_base+erlangshen_110m 0.7577  0.7120
                             macbert_bas

分類 dev-oracle / erlangshen_330m: 100%|██████████| 188/188 [01:13<00:00,  2.57it/s]


{
  "page_recall_at5": 0.9175,
  "page_recall_at_topk": 0.9245,
  "evidence_recall_at5": 0.8615,
  "evidence_recall_at5_no_hop": 0.8615,
  "hop_delta": null,
  "classification_acc_no_calibration": 0.7723333333333333,
  "classification_acc_old_prior_correction": 0.7753333333333333,
  "classification_acc_calibrated": 0.7753333333333333,
  "classification_acc_oracle_evidence": 0.796,
  "active_members": [
    "macbert_base",
    "roberta_base",
    "erlangshen_110m",
    "erlangshen_330m"
  ],
  "member_dev_acc": {
    "macbert_base": 0.7136666666666667,
    "roberta_base": 0.7226666666666667,
    "erlangshen_110m": 0.747,
    "erlangshen_330m": 0.7663333333333333
  },
  "ensemble_subsets": [
    {
      "members": "roberta_base+erlangshen_110m+erlangshen_330m",
      "acc": 0.7783,
      "cfever": 0.7327
    },
    {
      "members": "macbert_base+erlangshen_110m+erlangshen_330m",
      "acc": 0.776,
      "cfever": 0.73
    },
    {
      "members": "erlangshen_110m+erlangshen_330m",
  

## 7. 產生 submission.csv

格式與 sample code 相同：`id, predicted_label, predicted_evidence`；預測 NEI 的證據填 `[]`，其餘填前 5 句 `[page_id, sentence_id]`。

In [31]:
test_logits = classify(test, EV["test"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def make_submission(pred, path):
    evs = ["[]" if p == 2 else json.dumps([[titles[c[1]], c[2]] for c in EV["test"][qi][:5]], ensure_ascii=False) for qi, p in enumerate(pred)]
    out = pd.DataFrame({"id": test["id"], "predicted_label": [LABEL_OUT[p] for p in pred], "predicted_evidence": evs})
    out.to_csv(path, index=False, encoding="utf-8")
    return out


_sfx = "_quick" if QUICK_TEST else ""
sub_path = OUTPUT_DIR / f"submission{_sfx}.csv"                 # 正式版（若採用校準，則含校準）
submission = make_submission((test_logits + CAL_BIAS).argmax(1), sub_path)
sub_nocal = make_submission(test_logits.argmax(1), OUTPUT_DIR / f"submission_nocal{_sfx}.csv")   # 未校準版，可上傳 Kaggle 做 A/B 比較
(OUTPUT_DIR / ("metrics_quick.json" if QUICK_TEST else "metrics.json")).write_text(
    json.dumps(dict(metrics, per_label_recall=per_label, page_table=page_df.round(4).to_dict(), evidence_table=ev_df.round(4).to_dict()),
               indent=2, ensure_ascii=False), encoding="utf-8")
print("輸出筆數：", len(submission), "| 輸出位置：", sub_path.resolve())
print("預測標籤分佈（正式版）：", submission["predicted_label"].value_counts().to_dict())
print("預測標籤分佈（未校準）：", sub_nocal["predicted_label"].value_counts().to_dict())
# 額外分析：test 內有 3 種來源明顯不同的題目（見 Report 4(d)）：
#   A. id<=31996：與 train/dev 的 id 交錯，分佈類似 dev；
#   B. id>31996 且含「最喜歡」（1000 題）：虛構的個人偏好句，train/dev 完全沒有這種模板；
#   C. id>31996 其餘（2000 題）：長句、整句像是直接抄自維基。
_ids = np.array(test["id"])
_fav = np.array(["最喜歡" in c for c in test["claim"]])
groups = {"A id<=31996": _ids <= 31996, "B id>31996 含『最喜歡』": (_ids > 31996) & _fav, "C id>31996 其他": (_ids > 31996) & ~_fav}
for _name, _m in groups.items():
    print(f"{_name:24s} n={int(_m.sum()):4d} 預測分佈：", submission.loc[_m, "predicted_label"].value_counts(normalize=True).round(3).to_dict())
# 固定 NEI 偏差的備選提交檔（只動 NEI 的 logit；v1 的校準就是 NEI +0.75，且在 public 有進步）；最多只能選 2 份提交，請記錄每份的分數
for _b in (0.5, 1.0):
    if abs(_b - CAL_BIAS[2]) > 1e-9:
        make_submission((test_logits + np.array([0.0, 0.0, _b])).argmax(1), OUTPUT_DIR / f"submission_nei{_b}{_sfx}.csv")
submission.head()

分類 test / erlangshen_330m:  25%|██▌       | 94/375 [00:23<01:13,  3.84it/s]

分類 test / erlangshen_330m: 100%|██████████| 375/375 [02:22<00:00,  2.64it/s]


輸出筆數： 6000 | 輸出位置： /home/yiting/NLP-Assignment3/output/submission.csv
預測標籤分佈（正式版）： {'NOT ENOUGH INFO': 2126, 'refutes': 1960, 'supports': 1914}
預測標籤分佈（未校準）： {'supports': 2057, 'NOT ENOUGH INFO': 2011, 'refutes': 1932}
A id<=31996              n=3000 預測分佈： {'NOT ENOUGH INFO': 0.37, 'refutes': 0.327, 'supports': 0.303}
B id>31996 含『最喜歡』        n=1000 預測分佈： {'NOT ENOUGH INFO': 1.0}
C id>31996 其他            n=2000 預測分佈： {'supports': 0.502, 'refutes': 0.489, 'NOT ENOUGH INFO': 0.008}


,id,predicted_label,predicted_evidence
0,29579,NOT ENOUGH INFO,[]
1,3895,supports,"[[""鄱陽湖"", 0], [""江西省"", 1], [""鄱陽"", 3], [""鄱陽湖"", 2]..."
2,24522,refutes,"[[""赫伯特·胡佛"", 1], [""胡佛"", 5], [""赫伯特·胡佛"", 0], [""胡佛..."
3,11060,NOT ENOUGH INFO,[]
4,23607,supports,"[[""馬肯森級大巡洋艦"", 7], [""馬肯森級大巡洋艦"", 8], [""馬肯森號大巡洋艦""..."


In [32]:
import numpy as np
ids = np.array(test["id"]); fav = np.array(["最喜歡" in c for c in test["claim"]])
grp = {"A": ids <= 31996, "B": (ids > 31996) & fav, "C": (ids > 31996) & ~fav}
for b in (0, 0.5, 1.0, 1.5, 2.0, 3.0):
    pred = (test_logits + np.array([0.0, 0.0, b])).argmax(1)
    share = {g: round(float((pred[m] == 2).mean()), 3) for g, m in grp.items()}
    dev_cf = cf((dev_logits + np.array([0.0, 0.0, b])).argmax(1), dev_y, dev_ev_ok)
    print(f"NEI偏差 {b}: test 各組 NEI 比例 {share} | 全部 {float((pred == 2).mean()):.3f} | dev CFEVER {dev_cf:.4f}")
    if b in (1.5, 2.0, 3.0):
        make_submission(pred, OUTPUT_DIR / f"submission_nei{b}.csv")

NEI偏差 0: test 各組 NEI 比例 {'A': 0.332, 'B': 1.0, 'C': 0.008} | 全部 0.335 | dev CFEVER 0.7280
NEI偏差 0.5: test 各組 NEI 比例 {'A': 0.37, 'B': 1.0, 'C': 0.009} | 全部 0.354 | dev CFEVER 0.7327
NEI偏差 1.0: test 各組 NEI 比例 {'A': 0.42, 'B': 1.0, 'C': 0.009} | 全部 0.380 | dev CFEVER 0.7317
NEI偏差 1.5: test 各組 NEI 比例 {'A': 0.484, 'B': 1.0, 'C': 0.009} | 全部 0.412 | dev CFEVER 0.7193
NEI偏差 2.0: test 各組 NEI 比例 {'A': 0.555, 'B': 1.0, 'C': 0.011} | 全部 0.448 | dev CFEVER 0.6873
NEI偏差 3.0: test 各組 NEI 比例 {'A': 0.698, 'B': 1.0, 'C': 0.051} | 全部 0.533 | dev CFEVER 0.6040
